# Lid-Driven Cavity Flow con PINNs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/navier_stokes_pinns/notebooks/02_cavity.ipynb)

## 📚 Introducción

El **Lid-Driven Cavity** es un benchmark clásico de CFD:

- Cavidad cuadrada con todas las paredes fijas **excepto** la tapa superior
- La tapa se mueve con velocidad constante $U_{lid}$
- Genera uno o múltiples vórtices dependiendo del Re

**Diferencia clave vs Poiseuille:** No hay solución analítica conocida. Validamos contra resultados numéricos de referencia (Ghia et al., 1982).

## 🧮 Ecuaciones de Navier-Stokes 2D

**Conservación de momento:**

$$
u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} = -\frac{1}{\rho}\frac{\partial p}{\partial x} + \nu\left(\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}\right)
$$

$$
u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y} = -\frac{1}{\rho}\frac{\partial p}{\partial y} + \nu\left(\frac{\partial^2 v}{\partial x^2} + \frac{\partial^2 v}{\partial y^2}\right)
$$

**Continuidad (incompresibilidad):**

$$
\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} = 0
$$

### Condiciones de Frontera

- **Tapa (y = 1):** $u = U_{lid}$, $v = 0$
- **Otras paredes:** $u = 0$, $v = 0$ (no-slip)

## 🔧 Setup

In [ ]:
# Descomenta si estás en Google Colab
# !pip install deepxde tensorflow numpy matplotlib seaborn -q

In [ ]:
import sys
import os

if os.path.exists('../src'):
    sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import deepxde as dde

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Librerías cargadas")

## 🧮 Parámetros del Problema

Empezamos con **Re = 100** (flujo laminar con vórtice primario bien definido).

In [ ]:
# Geometría
L = 1.0          # Lado de la cavidad [m]

# Cinemática
U_lid = 1.0      # Velocidad de la tapa [m/s]
Re = 100.0       # Número de Reynolds

# Derivados
nu = U_lid * L / Re  # Viscosidad cinemática [m²/s]

print(f"Parámetros:")
print(f"  Lado de la cavidad: L = {L} m")
print(f"  Velocidad de la tapa: U = {U_lid} m/s")
print(f"  Número de Reynolds: Re = {Re}")
print(f"  Viscosidad cinemática: ν = {nu:.6f} m²/s")

## 🏗️ Construir el PINN

Ahora la red tiene **3 salidas**: $[u, v, p]$

In [ ]:
def navier_stokes(x, y):
    """
    Sistema completo de Navier-Stokes 2D.
    
    Args:
        x: [x, y] coordenadas
        y: [u, v, p] salidas de la red
    
    Returns:
        [residuo_continuidad, residuo_momento_x, residuo_momento_y]
    """
    u = y[:, 0:1]
    v = y[:, 1:2]
    p = y[:, 2:3]
    
    # Derivadas de u
    u_x = dde.grad.jacobian(y, x, i=0, j=0)
    u_y = dde.grad.jacobian(y, x, i=0, j=1)
    u_xx = dde.grad.hessian(y, x, component=0, i=0, j=0)
    u_yy = dde.grad.hessian(y, x, component=0, i=1, j=1)
    
    # Derivadas de v
    v_x = dde.grad.jacobian(y, x, i=1, j=0)
    v_y = dde.grad.jacobian(y, x, i=1, j=1)
    v_xx = dde.grad.hessian(y, x, component=1, i=0, j=0)
    v_yy = dde.grad.hessian(y, x, component=1, i=1, j=1)
    
    # Derivadas de presión
    p_x = dde.grad.jacobian(y, x, i=2, j=0)
    p_y = dde.grad.jacobian(y, x, i=2, j=1)
    
    # Ecuación de continuidad: ∂u/∂x + ∂v/∂y = 0
    continuity = u_x + v_y
    
    # Momento en x
    momentum_x = u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
    
    # Momento en y
    momentum_y = u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)
    
    return [continuity, momentum_x, momentum_y]

print("✓ PDE definida")

In [ ]:
# Geometría: cavidad [0, L] x [0, L]
geom = dde.geometry.Rectangle([0, 0], [L, L])

# Condiciones de frontera
def boundary_wall(x, on_boundary):
    """Paredes laterales e inferior"""
    return on_boundary and not np.isclose(x[1], L)

def boundary_lid(x, on_boundary):
    """Tapa superior"""
    return on_boundary and np.isclose(x[1], L)

# Tapa: u = U_lid, v = 0
bc_lid_u = dde.icbc.DirichletBC(geom, lambda x: U_lid, boundary_lid, component=0)
bc_lid_v = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_lid, component=1)

# Paredes: u = 0, v = 0
bc_wall_u = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_wall, component=0)
bc_wall_v = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_wall, component=1)

print("✓ Condiciones de frontera definidas")

In [ ]:
# Problema PDE
data = dde.data.PDE(
    geom,
    navier_stokes,
    [bc_lid_u, bc_lid_v, bc_wall_u, bc_wall_v],
    num_domain=5000,   # Más puntos que Poiseuille (problema más complejo)
    num_boundary=400,
    num_test=1000
)

# Red neuronal: [x, y] → [u, v, p]
net = dde.nn.FNN(
    [2, 50, 50, 50, 50, 3],  # 4 capas ocultas, 3 outputs
    "tanh",
    "Glorot normal"
)

model = dde.Model(data, net)

print("✓ Modelo construido")

## 🎯 Entrenar

**Nota:** Este entrenamiento toma varios minutos. ☕

In [ ]:
# Fase 1: Adam
model.compile("adam", lr=1e-3)
losshistory, train_state = model.train(iterations=20000, display_every=1000)

# Fase 2: L-BFGS
model.compile("L-BFGS")
losshistory, train_state = model.train()

print("\n✓ Entrenamiento completado")

## 📊 Convergencia

In [ ]:
dde.saveplot(losshistory, train_state, issave=False, isplot=True)
plt.show()

## 🌊 Visualizar Resultados

In [ ]:
# Crear malla de evaluación
x = np.linspace(0, L, 150)
y = np.linspace(0, L, 150)
X, Y = np.meshgrid(x, y)

# Predecir
points = np.column_stack([X.flatten(), Y.flatten()])
output = model.predict(points)

u = output[:, 0].reshape(X.shape)
v = output[:, 1].reshape(X.shape)
p = output[:, 2].reshape(X.shape)

# Calcular magnitud y vorticidad
vel_mag = np.sqrt(u**2 + v**2)

dx = x[1] - x[0]
dy = y[1] - y[0]
dvdx = np.gradient(v, dx, axis=1)
dudy = np.gradient(u, dy, axis=0)
vorticity = dvdx - dudy

print("✓ Campos calculados")

### Campo de Velocidad con Streamlines

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Magnitud de velocidad
contour = ax.contourf(X, Y, vel_mag, levels=30, cmap='viridis')
plt.colorbar(contour, ax=ax, label='|v| [m/s]')

# Streamlines
ax.streamplot(X, Y, u, v, color='white', linewidth=0.8, density=2, arrowsize=1.2)

ax.set_xlabel('x [m]', fontsize=12)
ax.set_ylabel('y [m]', fontsize=12)
ax.set_title(f'Lid-Driven Cavity - Re = {int(Re)}', fontsize=14, fontweight='bold')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

### Vorticidad (Rotación del Fluido)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

vmax = np.abs(vorticity).max()
contour = ax.contourf(X, Y, vorticity, levels=40, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
plt.colorbar(contour, ax=ax, label='ω [1/s]')

ax.set_xlabel('x [m]', fontsize=12)
ax.set_ylabel('y [m]', fontsize=12)
ax.set_title('Vorticidad', fontsize=14, fontweight='bold')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

### Presión

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

contour = ax.contourf(X, Y, p, levels=30, cmap='plasma')
plt.colorbar(contour, ax=ax, label='p [Pa]')

ax.set_xlabel('x [m]', fontsize=12)
ax.set_ylabel('y [m]', fontsize=12)
ax.set_title('Campo de Presión', fontsize=14, fontweight='bold')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 📈 Perfiles en Líneas Centrales

Estos se comparan típicamente con datos de referencia (Ghia et al., 1982).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Perfil u en línea vertical central (x = 0.5)
idx_x_center = len(x) // 2
u_vertical = u[:, idx_x_center]

ax1.plot(u_vertical, y, 'b-', linewidth=2, label='u (x=0.5)')
ax1.axhline(L/2, color='gray', linestyle='--', alpha=0.3)
ax1.axvline(0, color='gray', linestyle='--', alpha=0.3)
ax1.set_xlabel('Velocidad u [m/s]', fontsize=12)
ax1.set_ylabel('y [m]', fontsize=12)
ax1.set_title('Perfil de u en Línea Central Vertical', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Perfil v en línea horizontal central (y = 0.5)
idx_y_center = len(y) // 2
v_horizontal = v[idx_y_center, :]

ax2.plot(x, v_horizontal, 'r-', linewidth=2, label='v (y=0.5)')
ax2.axvline(L/2, color='gray', linestyle='--', alpha=0.3)
ax2.axhline(0, color='gray', linestyle='--', alpha=0.3)
ax2.set_xlabel('x [m]', fontsize=12)
ax2.set_ylabel('Velocidad v [m/s]', fontsize=12)
ax2.set_title('Perfil de v en Línea Central Horizontal', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## 🎯 Posición del Centro del Vórtice

Para Re=100, el centro del vórtice primario debería estar aproximadamente en (x, y) ≈ (0.62, 0.74) según Ghia et al.

In [ ]:
# Encontrar mínimo de función de corriente (centro del vórtice)
from scipy.integrate import cumtrapz

# Integrar v en x para obtener función de corriente
psi = np.zeros_like(u)
for i in range(u.shape[0]):
    psi[i, :] = cumtrapz(v[i, :], x, initial=0)

# Encontrar mínimo
min_idx = np.unravel_index(np.argmin(psi), psi.shape)
x_vortex = X[min_idx]
y_vortex = Y[min_idx]

print(f"Centro del vórtice predicho:")
print(f"  x = {x_vortex:.4f} m")
print(f"  y = {y_vortex:.4f} m")
print(f"\nReferencia (Ghia et al., Re=100):")
print(f"  x ≈ 0.6172")
print(f"  y ≈ 0.7372")

## 🎓 Conclusiones

**¿Qué logramos?**

1. ✅ Resolvimos el sistema completo de Navier-Stokes 2D sin solución analítica
2. ✅ Capturamos el vórtice primario característico
3. ✅ Obtuvimos campos de velocidad, presión y vorticidad físicamente consistentes

**Comparado con CFD tradicional:**

- ❌ Menos preciso para validación industrial
- ✅ No requiere mallado manual
- ✅ Puede incorporar datos experimentales escasos directamente en el loss

**Experimentos adicionales:**

- Cambiar Re a 400, 1000 (vórtices secundarios)
- Aumentar capas/neuronas para mejorar precisión
- Comparar cuantitativamente con datos de Ghia